# ColabPLM — fine-tuning any PLM with LoRA inside Colab

This notebook adapts the **ColabSaprot** template into a model-agnostic
**ColabPLM**: a one-stop pipeline that lets you fine-tune any
HuggingFace-hosted Protein Language Model (ESM-2, ESM-C, ESM-3, ProtT5, SaProt…)
on a sequence-classification or regression task using parameter-efficient
LoRA adapters.

Sections:
1. Environment & GPU check
2. Install dependencies
3. Choose backbone & task
4. Data preparation (DeepLoc binary membrane vs soluble)
5. Build model + LoRA adapter
6. Train
7. Save and re-load the adapter
8. Inference demo


## 1. Environment & GPU check

In [ ]:
!nvidia-smi -L || echo 'no GPU — switch to GPU runtime via Runtime > Change runtime type'
import platform, sys
print('python', sys.version.split()[0], '|', platform.platform())

## 2. Install dependencies

In [ ]:
%pip install -q transformers==4.46.3 peft==0.13.2 pytorch-lightning==2.4.0 \
                  datasets scikit-learn pandas biopython safetensors

## 3. Choose a backbone PLM and downstream task

Pick one of the entries below (or paste any HuggingFace seq-classification-capable PLM identifier). LoRA target modules are auto-detected per family.

In [ ]:
MODEL_NAME = "facebook/esm2_t12_35M_UR50D"   # @param ['facebook/esm2_t6_8M_UR50D','facebook/esm2_t12_35M_UR50D','facebook/esm2_t30_150M_UR50D','facebook/esm2_t33_650M_UR50D','Rostlab/prot_t5_xl_uniref50','westlake-repl/SaProt_35M_AF2'] {allow-input: true}
TASK = "deeploc_binary"   # membrane (1) vs soluble (0)
NUM_LABELS  = 2
IS_REGRESSION = False
MAX_LENGTH = 256
RANK, ALPHA = 8, 16
EPOCHS, BATCH_SIZE, LR = 3, 16, 5e-4

## 4. Fetch the ColabPLM source modules

We ship the ColabPLM library as plain `.py` files inline so the notebook is self-contained on Colab without needing a separate `git clone`.

In [ ]:
import os, textwrap, pathlib
pathlib.Path('colab_plm').mkdir(exist_ok=True)
SRC = "# -- colab_plm/loader.py --\n\"\"\"Backbone loader for ColabPLM.\n\nA unified factory that hides the per-PLM quirks (ProtT5 needs spaced tokens,\nESM-2 accepts raw strings, ESM-C/ESM3 may need trust_remote_code, etc.) so the\nrest of the pipeline can stay model-agnostic.\n\"\"\"\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom typing import Tuple\n\nimport torch\nfrom transformers import AutoModelForSequenceClassification, AutoTokenizer\n\n\n@dataclass\nclass BackboneBundle:\n    tokenizer: object\n    model: torch.nn.Module\n    is_t5: bool\n    family: str   # \"esm2\" | \"esmc\" | \"esm3\" | \"prott5\" | \"saprot\" | \"generic\"\n    hidden_size: int\n\n\ndef _detect_family(model_name: str) -> str:\n    n = model_name.lower()\n    if \"saprot\" in n:\n        return \"saprot\"\n    if \"esm3\" in n or \"esm-3\" in n:\n        return \"esm3\"\n    if \"esmc\" in n or \"esm-c\" in n or \"esm_c\" in n:\n        return \"esmc\"\n    if \"esm2\" in n or \"esm-2\" in n or \"esm_2\" in n or \"esm\" in n:\n        return \"esm2\"\n    if \"prot_t5\" in n or \"prott5\" in n or \"t5\" in n:\n        return \"prott5\"\n    return \"generic\"\n\n\nclass ColabPLMLoader:\n    \"\"\"Load any seq-classification-capable PLM from a HF identifier or local path.\"\"\"\n\n    @staticmethod\n    def load_backbone(\n        model_name_or_path: str,\n        num_labels: int,\n        is_regression: bool = False,\n        cache_dir: str | None = None,\n    ) -> BackboneBundle:\n        family = _detect_family(model_name_or_path)\n        is_t5 = family == \"prott5\"\n\n        tokenizer = AutoTokenizer.from_pretrained(\n            model_name_or_path,\n            trust_remote_code=True,\n            cache_dir=cache_dir,\n            do_lower_case=False,\n        )\n\n        problem_type = \"regression\" if is_regression else \"single_label_classification\"\n        model = AutoModelForSequenceClassification.from_pretrained(\n            model_name_or_path,\n            num_labels=num_labels,\n            problem_type=problem_type,\n            trust_remote_code=True,\n            cache_dir=cache_dir,\n            ignore_mismatched_sizes=True,\n        )\n\n        hidden_size = getattr(model.config, \"hidden_size\", None) or getattr(\n            model.config, \"d_model\", -1\n        )\n\n        return BackboneBundle(\n            tokenizer=tokenizer,\n            model=model,\n            is_t5=is_t5,\n            family=family,\n            hidden_size=hidden_size,\n        )\n\n\n# -- colab_plm/data.py --\n\"\"\"Protein sequence data pipeline.\n\nHandles three concerns that the original ColabSaprot code hardcoded for SaProt:\n  * Spacing \u2014 ProtT5 expects whitespace-separated residues, ESM-2 does not.\n  * Sanitization \u2014 non-standard residues (U Z O B) must collapse to X before\n    tokenization or the tokenizer raises on unknown ids.\n  * Length \u2014 long sequences are truncated to ``max_length`` to keep the LoRA\n    fine-tune within Colab T4 memory.\n\"\"\"\nfrom __future__ import annotations\n\nfrom typing import Tuple\n\nimport pandas as pd\nimport torch\nfrom torch.utils.data import DataLoader, Dataset\n\n_STANDARD_AA = set(\"ACDEFGHIKLMNPQRSTVWY\")\n\n\ndef sanitize_sequence(seq: str) -> str:\n    seq = seq.upper().replace(\" \", \"\")\n    return \"\".join(c if c in _STANDARD_AA else \"X\" for c in seq)\n\n\nclass ProteinSequenceDataset(Dataset):\n    \"\"\"CSV-backed dataset where each row has ``protein`` and ``label`` columns.\"\"\"\n\n    def __init__(\n        self,\n        csv_path: str,\n        tokenizer,\n        max_length: int = 1024,\n        is_t5: bool = False,\n        is_regression: bool = False,\n    ):\n        self.df = pd.read_csv(csv_path)\n        assert \"protein\" in self.df.columns and \"label\" in self.df.columns, (\n            f\"CSV {csv_path} must contain 'protein' and 'label' columns\"\n        )\n        self.tokenizer = tokenizer\n        self.max_length = max_length\n        self.is_t5 = is_t5\n        self.label_dtype = torch.float32 if is_regression else torch.long\n\n    def __len__(self) -> int:\n        return len(self.df)\n\n    def __getitem__(self, idx: int):\n        row = self.df.iloc[idx]\n        seq = sanitize_sequence(str(row[\"protein\"]))\n        if self.is_t5:\n            seq = \" \".join(list(seq))\n\n        enc = self.tokenizer(\n            seq,\n            padding=\"max_length\",\n            truncation=True,\n            max_length=self.max_length,\n            return_tensors=\"pt\",\n        )\n        item = {k: v.squeeze(0) for k, v in enc.items()}\n        item[\"labels\"] = torch.tensor(row[\"label\"], dtype=self.label_dtype)\n        return item\n\n\ndef build_dataloaders(\n    tokenizer,\n    train_csv: str,\n    val_csv: str | None = None,\n    batch_size: int = 8,\n    max_length: int = 512,\n    is_t5: bool = False,\n    is_regression: bool = False,\n    num_workers: int = 2,\n) -> Tuple[DataLoader, DataLoader | None]:\n    train_ds = ProteinSequenceDataset(\n        train_csv, tokenizer, max_length, is_t5, is_regression\n    )\n    train_loader = DataLoader(\n        train_ds,\n        batch_size=batch_size,\n        shuffle=True,\n        num_workers=num_workers,\n        pin_memory=True,\n    )\n    val_loader = None\n    if val_csv is not None:\n        val_ds = ProteinSequenceDataset(\n            val_csv, tokenizer, max_length, is_t5, is_regression\n        )\n        val_loader = DataLoader(\n            val_ds,\n            batch_size=batch_size,\n            shuffle=False,\n            num_workers=num_workers,\n            pin_memory=True,\n        )\n    return train_loader, val_loader\n\n\n# -- colab_plm/adapter.py --\n\"\"\"LoRA injection.\n\nColabSaprot's adapter step assumes SaProt's attention naming.  Different PLM\nfamilies name their projection matrices differently, so we discover the right\n``target_modules`` from the model itself rather than hard-coding.\n\"\"\"\nfrom __future__ import annotations\n\nfrom typing import Iterable, List\n\nimport torch\nfrom peft import LoraConfig, TaskType, get_peft_model\n\n# Ordered by preference \u2014 first family of patterns that exists in the model wins.\n_PATTERN_GROUPS: List[List[str]] = [\n    [\"q_proj\", \"v_proj\"],                # ESM-C / ESM-3 / LLaMA-style\n    [\"query\", \"value\"],                  # ESM-2 / BERT / SaProt\n    [\"SelfAttention.q\", \"SelfAttention.v\"],  # ProtT5 / T5\n]\n\n\n_HEAD_PATTERNS = [\"classifier\", \"score\", \"classification_head\"]\n\n\ndef suggest_modules_to_save(model: torch.nn.Module) -> List[str]:\n    \"\"\"Identify the freshly-initialised classification head so PEFT persists it.\"\"\"\n    names = {n for n, _ in model.named_modules()}\n    out: list[str] = []\n    for p in _HEAD_PATTERNS:\n        if any(n == p or n.endswith(\".\" + p) for n in names):\n            out.append(p)\n    return out\n\n\ndef suggest_target_modules(model: torch.nn.Module) -> List[str]:\n    names = {n for n, _ in model.named_modules()}\n\n    def _present(pat: str) -> bool:\n        return any(n.endswith(pat) or n.endswith(\".\" + pat) for n in names)\n\n    for group in _PATTERN_GROUPS:\n        if all(_present(p) for p in group):\n            return group\n    raise RuntimeError(\n        \"Could not auto-detect LoRA target modules.  Pass target_modules explicitly.  \"\n        \"Sampled module names: \" + \", \".join(list(names)[:10])\n    )\n\n\ndef inject_lora_adapter(\n    model: torch.nn.Module,\n    target_modules: Iterable[str] | None = None,\n    rank: int = 8,\n    alpha: int = 16,\n    dropout: float = 0.1,\n    task_type: TaskType = TaskType.SEQ_CLS,\n):\n    if target_modules is None:\n        target_modules = suggest_target_modules(model)\n    modules_to_save = suggest_modules_to_save(model) or None\n    cfg = LoraConfig(\n        task_type=task_type,\n        r=rank,\n        lora_alpha=alpha,\n        lora_dropout=dropout,\n        target_modules=list(target_modules),\n        modules_to_save=modules_to_save,\n        bias=\"none\",\n    )\n    peft_model = get_peft_model(model, cfg)\n    peft_model.print_trainable_parameters()\n    return peft_model\n\n\n# -- colab_plm/trainer.py --\n\"\"\"Minimal training/eval/inference loops for ColabPLM.\"\"\"\nfrom __future__ import annotations\n\nimport os\nfrom typing import Iterable, List, Tuple\n\nimport torch\nfrom torch.optim import AdamW\nfrom torch.utils.data import DataLoader\nfrom sklearn.metrics import accuracy_score, f1_score, mean_squared_error\n\nfrom .loader import BackboneBundle, ColabPLMLoader\nfrom .data import build_dataloaders, sanitize_sequence\nfrom .adapter import inject_lora_adapter\n\n\ndef _device() -> torch.device:\n    return torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n\n\n@torch.no_grad()\ndef evaluate(model, loader: DataLoader, is_regression: bool = False):\n    model.eval()\n    device = next(model.parameters()).device\n    preds: List[float] = []\n    golds: List[float] = []\n    total_loss = 0.0\n    for batch in loader:\n        batch = {k: v.to(device) for k, v in batch.items()}\n        out = model(**batch)\n        total_loss += float(out.loss)\n        if is_regression:\n            preds.extend(out.logits.squeeze(-1).cpu().tolist())\n        else:\n            preds.extend(out.logits.argmax(-1).cpu().tolist())\n        golds.extend(batch[\"labels\"].cpu().tolist())\n    if is_regression:\n        metric = {\"mse\": mean_squared_error(golds, preds)}\n    else:\n        metric = {\n            \"acc\": accuracy_score(golds, preds),\n            \"f1\": f1_score(golds, preds, average=\"macro\"),\n        }\n    metric[\"loss\"] = total_loss / max(len(loader), 1)\n    return metric\n\n\ndef train_colab_plm(\n    model_name: str,\n    train_csv: str,\n    val_csv: str | None,\n    num_labels: int,\n    output_dir: str,\n    epochs: int = 3,\n    batch_size: int = 8,\n    lr: float = 5e-4,\n    rank: int = 8,\n    alpha: int = 16,\n    max_length: int = 512,\n    is_regression: bool = False,\n    target_modules: Iterable[str] | None = None,\n    cache_dir: str | None = None,\n    fp16: bool = True,\n    log_every: int = 5,\n) -> Tuple[torch.nn.Module, BackboneBundle, dict]:\n    \"\"\"One-shot orchestrator: load \u2192 inject LoRA \u2192 fine-tune \u2192 save.\"\"\"\n    device = _device()\n    print(f\"[ColabPLM] device={device}\")\n\n    bundle = ColabPLMLoader.load_backbone(\n        model_name, num_labels=num_labels,\n        is_regression=is_regression, cache_dir=cache_dir,\n    )\n    print(f\"[ColabPLM] family={bundle.family} hidden_size={bundle.hidden_size}\")\n\n    peft_model = inject_lora_adapter(\n        bundle.model, target_modules=target_modules, rank=rank, alpha=alpha,\n    ).to(device)\n\n    train_loader, val_loader = build_dataloaders(\n        bundle.tokenizer, train_csv, val_csv,\n        batch_size=batch_size, max_length=max_length,\n        is_t5=bundle.is_t5, is_regression=is_regression,\n    )\n\n    optimizer = AdamW(filter(lambda p: p.requires_grad, peft_model.parameters()), lr=lr)\n    scaler = torch.cuda.amp.GradScaler(enabled=fp16 and device.type == \"cuda\")\n\n    history = {\"train_loss\": [], \"val\": []}\n    for epoch in range(epochs):\n        peft_model.train()\n        running = 0.0\n        for step, batch in enumerate(train_loader):\n            batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}\n            optimizer.zero_grad(set_to_none=True)\n            with torch.cuda.amp.autocast(\n                enabled=fp16 and device.type == \"cuda\", dtype=torch.float16\n            ):\n                out = peft_model(**batch)\n                loss = out.loss\n            scaler.scale(loss).backward()\n            scaler.step(optimizer)\n            scaler.update()\n            running += float(loss)\n            if (step + 1) % log_every == 0:\n                print(f\"  epoch {epoch+1} step {step+1}/{len(train_loader)} loss={loss.item():.4f}\")\n        avg = running / max(len(train_loader), 1)\n        history[\"train_loss\"].append(avg)\n        msg = f\"[ColabPLM] epoch {epoch+1}/{epochs} train_loss={avg:.4f}\"\n        if val_loader is not None:\n            m = evaluate(peft_model, val_loader, is_regression)\n            history[\"val\"].append(m)\n            msg += \" | val=\" + \", \".join(f\"{k}={v:.4f}\" for k, v in m.items())\n        print(msg)\n\n    os.makedirs(output_dir, exist_ok=True)\n    peft_model.save_pretrained(output_dir)\n    bundle.tokenizer.save_pretrained(output_dir)\n    print(f\"[ColabPLM] adapter saved to {output_dir}\")\n    return peft_model, bundle, history\n\n\n@torch.no_grad()\ndef predict_sequences(\n    model, tokenizer, sequences: Iterable[str],\n    is_t5: bool = False, max_length: int = 512,\n):\n    model.eval()\n    device = next(model.parameters()).device\n    seqs = [sanitize_sequence(s) for s in sequences]\n    if is_t5:\n        seqs = [\" \".join(list(s)) for s in seqs]\n    enc = tokenizer(\n        seqs, padding=True, truncation=True,\n        max_length=max_length, return_tensors=\"pt\",\n    ).to(device)\n    logits = model(**enc).logits\n    return logits.cpu()\n\n\n"
for chunk in SRC.split('# -- colab_plm/'):
    if not chunk.strip():
        continue
    name, body = chunk.split(' --', 1)
    (pathlib.Path('colab_plm') / name).write_text(body.lstrip('\n'))
pathlib.Path('colab_plm/__init__.py').write_text('''
from .loader import ColabPLMLoader
from .data import ProteinSequenceDataset, build_dataloaders, sanitize_sequence
from .adapter import inject_lora_adapter, suggest_target_modules
from .trainer import train_colab_plm, evaluate, predict_sequences
''')
print('wrote colab_plm/', os.listdir('colab_plm'))

## 5. Build the dataset (DeepLoc binary)

In [ ]:
from datasets import load_dataset
import pandas as pd, os

os.makedirs('data', exist_ok=True)
ds = load_dataset('proteinea/deeploc')

def to_df(split, n):
    d = ds[split].to_pandas()
    d = d[d['membrane'].isin(['M','S'])]
    df = pd.DataFrame({'protein': d['input'].str[:1022], 'label': (d['membrane']=='M').astype(int)})
    return df.sample(n=min(n,len(df)), random_state=0).reset_index(drop=True)

train_df = to_df('train', 2000)
val_df   = to_df('test',  500)
train_df.to_csv('data/train.csv', index=False)
val_df.to_csv('data/val.csv',   index=False)
print('train', len(train_df), 'val', len(val_df))
train_df.head(3)

## 6. Train ESM-2 + LoRA

In [ ]:
from colab_plm import train_colab_plm, predict_sequences

model, bundle, history = train_colab_plm(
    model_name=MODEL_NAME,
    train_csv='data/train.csv', val_csv='data/val.csv',
    num_labels=NUM_LABELS, output_dir='outputs/colab_plm_adapter',
    epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR,
    rank=RANK, alpha=ALPHA, max_length=MAX_LENGTH,
    is_regression=IS_REGRESSION,
)
history

## 7. Reload the saved LoRA adapter (simulating a fresh runtime)

In [ ]:
import importlib, colab_plm
importlib.reload(colab_plm)
from colab_plm import ColabPLMLoader
from peft import PeftModel

bundle = ColabPLMLoader.load_backbone(MODEL_NAME, num_labels=NUM_LABELS, is_regression=IS_REGRESSION)
reloaded = PeftModel.from_pretrained(bundle.model, 'outputs/colab_plm_adapter').eval().cuda()
print('reloaded — trainable params:')
reloaded.print_trainable_parameters()

## 8. Inference demo

In [ ]:
demo = [
    'MKTIIALSYIFCLVFADYKDDDDK',                        # short signal peptide
    'MFLVLLALVAVSAQGNCKEHLTKLPELPSGCTKVTLQ' * 3,        # soluble-ish
    'FFLLAAVVVLLIIVVAAFFLLAVVI' * 6,                    # hydrophobic, membrane-like
]
logits = predict_sequences(reloaded, bundle.tokenizer, demo, is_t5=bundle.is_t5, max_length=MAX_LENGTH)
print('logits:'); print(logits)
print('predictions (0=soluble, 1=membrane):', logits.argmax(-1).tolist())

## Notes
* Switching to ESM-C, ESM-3 or ProtT5 only requires changing ``MODEL_NAME``; the
  loader picks the correct tokenizer (with whitespace separation for T5) and
  the adapter builder auto-detects ``target_modules``.
* For regression tasks set ``IS_REGRESSION=True`` and ``NUM_LABELS=1``; the
  ``problem_type='regression'`` flag flows through automatically.
* The saved ``outputs/colab_plm_adapter`` directory contains only the LoRA
  matrices + the classification head (~1.6 MB for ESM-2-35M); the frozen
  backbone is fetched from HuggingFace on demand.